# 🏠 Aula 1 — Exercícios para Casa
**Pesquisa Operacional e Otimização em IA** — MBA em Ciência de Dados (UNIFOR)

Prof. Mafra | mafra@verboo.ai

---

Dois problemas para resolver usando PuLP. Use o notebook de demonstração da aula como referência.

### Biblioteca recomendada

- **PuLP** — modelagem e resolução de Programação Linear e Inteira
  - Documentação: https://coin-or.github.io/pulp/
  - Guia rápido: https://realpython.com/linear-programming-python/

In [2]:
!pip install pulp -q
from pulp import *


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


---
## Exercício 1 — Budget de Marketing 📈

Uma startup de tecnologia tem um budget de marketing de **R$ 50 mil por mês** e pode investir em três canais: **Google Ads** (gera 40 leads por R$ 1k investido, limite de R$ 30k), **Instagram** (65 leads por R$ 1k, limite de R$ 20k) e **Eventos** (10 leads por R$ 1k, limite de R$ 15k).

Existe uma restrição adicional: o investimento em Instagram e Eventos **juntos** não pode ultrapassar R$ 25 mil.

**Objetivo:** maximizar o número total de leads gerados.

### O que fazer

1. Identifique as **variáveis de decisão** (dica: use R$ 1k como unidade)
2. Escreva a **função objetivo**
3. Liste todas as **restrições** (são 5)
4. Modele e resolva com PuLP
5. Analise: quais restrições esgotaram? Algum canal ficou zerado? Por quê?


| Canal | Custo/mês (R$) | Qtd Leads | Limite (R$) |
|---|---|---|---|
| Google Ads | 1k | 40 | 30k |
| Instagram | 1k | 65 | 20k |
| Eventos | 1k | 10 | 15k |

**Budget:** R$ 50.000/mês

**Restrições:**
- O investimento em Instagram e Eventos **juntos** não pode ultrapassar R$ 25 mil.

**Objetivo:** maximizar o número total de leads gerados.

In [4]:
# Exercício 1 — escreva seu código aqui
prob1 = LpProblem("Canais_de_Vendas", LpMaximize)

x1 = LpVariable("Google_Ads", lowBound=0)
x2 = LpVariable("Instagram", lowBound=0)
x3 = LpVariable("Eventos", lowBound=0)

prob1 += 40*x1 + 65*x2 + 10*x3, "Leads"
prob1 += x1 + x2 + x3 <= 50, "Budget"
prob1 += x1 <= 30, "Limite_Google"
prob1 += x2 <= 20, "Limite_Instagram"
prob1 += x3 <= 15, "Limite_Eventos"
prob1 += x2 + x3 <= 25, "Limite_Insta_Eventos"

prob1.solve(PULP_CBC_CMD(msg=0))

print(f"Status: {LpStatus[prob1.status]}")
print(f"Google Ads: R$ {x1.varValue:.0f}k")
print(f"Instagram: R$ {x2.varValue:.0f}k")
print(f"Eventos: R$ {x3.varValue:.0f}k")
print(f"Leads gerados: {value(prob1.objective):.0f}")

Status: Optimal
Google Ads: R$ 30k
Instagram: R$ 20k
Eventos: R$ 0k
Leads gerados: 2500


---
## Exercício 2 — Escala Hospitalar 🏥

Um hospital precisa montar a escala semanal de **8 médicos** distribuídos em **3 turnos** (manhã, tarde e noite) ao longo de **5 dias úteis**.

As regras são:
- Cada turno precisa de pelo menos **2 médicos**
- Nenhum médico pode trabalhar mais de **5 turnos** na semana
- Quem faz o turno da **noite** não pode fazer o turno da **manhã no dia seguinte** (descanso obrigatório)
- Cada médico tem uma nota de preferência de 1 a 10 por turno — uns preferem manhã, outros noite

**Objetivo:** montar a escala que maximize a satisfação total dos médicos.

### O que fazer

1. Defina as **variáveis de decisão** (dica: são binárias — médico i trabalha no turno j do dia k?)
2. Quantas variáveis o modelo tem?
3. Escreva a **função objetivo** usando as preferências
4. Modele cada restrição (mínimo por turno, máximo por médico, descanso noite→manhã)
5. Invente uma matriz de preferências (ou use números aleatórios)
6. Resolva com PuLP e analise: algum médico ficou num turno que não gosta?
7. Qual sua análise quanto o problema?

### Dicas

- Use `cat="Binary"` — cada variável é 0 ou 1
- Pra criar muitas variáveis de uma vez: `LpVariable.dicts("x", (medicos, turnos, dias), cat="Binary")`
- A restrição de descanso liga o turno noite do dia `d` com o turno manhã do dia `d+1`
- Documentação de `LpVariable.dicts`: https://coin-or.github.io/pulp/guides/how_to_configure_solvers.html

In [18]:
import numpy as np

prob2 = LpProblem("Escala_Hospitalar", LpMaximize)

np.random.seed(42)

medicos = range(8)
turnos = ["Manha", "Tarde", "Noite"]
dias = range(5)

pref = np.random.randint(1, 11, size=(8, 3, 5))

x = LpVariable.dicts("escala", (medicos, turnos, dias), cat="Binary")

prob2 += lpSum(pref[i][t][d] * x[i][turnos[t]][d]
for i in medicos
for t, _ in enumerate(turnos)
for d in dias), "Satisfacao"

for d in dias:
    for t in turnos:
        prob2 += lpSum(x[m][t][d] for m in medicos) >= 2

for m in medicos:
    prob2 += pulp.lpSum(x[m][t][d] for t in turnos for d in dias) <= 5

for m in medicos:
    for d in range(4): # De segunda (1) a quinta (4)
        prob2 += x[m]["Noite"][d] + x[m]["Manha"][d+1] <= 1

prob2.solve()

print(f"Status: {LpStatus[prob2.status]}")
print(f"Satisfação Total: {value(prob2.objective)}")

for i in medicos:
    print(f"\nMédico {i}:")
    for k in dias:
        for j in turnos:
            if value(x[i][j][k]) == 1:
                print(f"  Dia {k} - {j}")

for d in dias:
    print(f"\nDia {d}:")
    for t in turnos:
        alocados = [m for m in medicos if value(x[m][t][d]) == 1]
        print(f"  {t}: {alocados}")

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /Users/dudanobre/venvs/data-science/lib/python3.13/site-packages/pulp/apis/../solverdir/cbc/osx/i64/cbc /var/folders/gw/38t0m3710p57vxqjw00sb8q00000gn/T/3c2c953dd1e14f6dab367eda84a31203-pulp.mps -max -timeMode elapsed -branch -printingOptions all -solution /var/folders/gw/38t0m3710p57vxqjw00sb8q00000gn/T/3c2c953dd1e14f6dab367eda84a31203-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 60 COLUMNS
At line 725 RHS
At line 781 BOUNDS
At line 902 ENDATA
Problem MODEL has 55 rows, 120 columns and 304 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is 341 - 0.00 seconds
Cgl0004I processed model has 55 rows, 120 columns (120 integer (120 of which binary)) and 304 elements
Cutoff increment increased from 1e-05 to 0.9999
Cbc0038I Initial state - 0 integers unsatisfied sum - 0
Cbc0038I Solution found of -

---

### Referências úteis

| Recurso | Link |
|---------|------|
| Documentação PuLP | https://coin-or.github.io/pulp/ |
| Tutorial PuLP (Real Python) | https://realpython.com/linear-programming-python/ |
| Exemplos PuLP (GitHub) | https://github.com/coin-or/pulp/tree/master/examples |
| Curso de PO (YouTube, em PT) | Pesquise "Pesquisa Operacional PuLP Python" |

**Entrega:** enviem o notebook (.ipynb) com o código funcionário até a próxima aula.